In [1]:
# ── Cell 0: HuggingFace login ──────────────────────────────────────────────
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN1 = user_secrets.get_secret("HF_TOKEN1")
login(token=HF_TOKEN1)
print("HF login: OK")

HF login: OK


In [2]:
# ── Cell 1: Install ────────────────────────────────────────────────────────
!pip install peft bitsandbytes transformers accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [3]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────
import os, re, time, warnings
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
warnings.filterwarnings("ignore")
print("Imports: OK")

Imports: OK


In [4]:
# ── Cell 3: Config ─────────────────────────────────────────────────────────
TRAIN_TXT  = "/kaggle/input/datasets/softhell/med-vqa-2021/data/train/label.txt"
TRAIN_IMGS = "/kaggle/input/datasets/softhell/med-vqa-2021/data/train/images"
VAL_TXT    = "/kaggle/input/datasets/softhell/med-vqa-2021/data/val/label.txt"
VAL_IMGS   = "/kaggle/input/datasets/softhell/med-vqa-2021/data/val/images"
OUTPUT_DIR = "/kaggle/working/qwen_vqa_lora"
EVAL_CSV   = "/kaggle/working/finetuned_model_metrics.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID   = "Qwen/Qwen2.5-VL-7B-Instruct"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# Vision token budget — keep small for T4 (50k tokens/image at 128*28*28)
MIN_PIXELS = 64  * 28 * 28
MAX_PIXELS = 128 * 28 * 28

# LoRA
LORA_RANK      = 32
LORA_ALPHA     = 64
LORA_DROPOUT   = 0.01
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# Training
LEARNING_RATE  = 2e-4
NUM_EPOCHS     = 5
GRAD_ACCUM     = 16      # effective batch = 16
WARMUP_STEPS   = 50
MAX_NEW_TOKENS = 20

SYSTEM_PROMPT = (
    "You are a radiology AI assistant. "
    "When asked about a medical image, respond with the diagnosis "
    "or abnormality name only — no explanations, no full sentences. "
    "Keep your answer to 1-6 words maximum."
)

print(f"Device : {DEVICE}")
print(f"Output : {OUTPUT_DIR}")

Device : cuda
Output : /kaggle/working/qwen_vqa_lora


In [5]:
# ── Cell 4: Helpers ────────────────────────────────────────────────────────
def normalise(text):
    """Lowercase, collapse whitespace."""
    return re.sub(r"\s+", " ", str(text).strip().lower())


def get_synonyms(answer):
    """Split comma-separated GT variants and normalise each."""
    return [normalise(s) for s in str(answer).split(",") if normalise(s)]


def exact_match_any(pred, answer):
    """Return 1 if pred matches any GT synonym."""
    return int(normalise(pred) in get_synonyms(answer))


def find_subseq(haystack, needle):
    """
    Return the START index of the LAST occurrence of `needle` (list)
    inside `haystack` (list).  Returns None if not found.
    Searching from the right handles the case where the answer tokens
    also appear earlier in the prompt (e.g. question text echoes the label).
    """
    n, h = len(needle), len(haystack)
    for i in range(h - n, -1, -1):
        if haystack[i:i + n] == needle:
            return i
    return None


print("Helpers: OK")


Helpers: OK


In [6]:
# ── Cell 5: Dataset ────────────────────────────────────────────────────────
class VQAMedDataset(Dataset):
    """
    Reads the pipe-delimited VQA-Med 2021 label files.
    The answer stored in self.records is the FIRST canonical synonym
    (used as the training target).  The raw_answer (all synonyms) is
    also stored for evaluation.
    """
    def __init__(self, txt_path, img_dir, split="train"):
        df = pd.read_csv(txt_path, sep="|",
                         names=["image_id", "question", "answer"], header=None)
        self.records = []
        for _, row in df.iterrows():
            img_id = str(row["image_id"]).strip()
            fname  = img_id if img_id.endswith((".jpg", ".png")) else img_id + ".jpg"
            path   = os.path.join(img_dir, fname)
            if not os.path.exists(path):
                continue
            synonyms = get_synonyms(str(row["answer"]))
            if not synonyms:
                continue
            self.records.append({
                "img_path":   path,
                "question":   str(row["question"]).strip(),
                "answer":     synonyms[0],      # canonical training target
                "raw_answer": str(row["answer"]).strip(),  # all synonyms for eval
            })
        print(f"  [{split}] Loaded {len(self.records)} samples")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            "image":      Image.open(r["img_path"]).convert("RGB"),
            "question":   r["question"],
            "answer":     r["answer"],
            "raw_answer": r["raw_answer"],
        }

In [7]:
# ── Cell 6: Collate — FIXED label masking ─────────────────────────────────
def make_collate(processor):
    def collate(batch):
        texts, images = [], []
        for s in batch:
            msgs = [
                {"role": "system",
                 "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
                {"role": "user",
                 "content": [{"type": "image", "image": s["image"]},
                              {"type": "text",  "text": "Question: " + s["question"]}]},
                {"role": "assistant",
                 "content": [{"type": "text",  "text": s["answer"]}]},
            ]
            texts.append(processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False))
            images.append(s["image"])

        enc = processor(
            text=texts, images=images,
            padding=True, return_tensors="pt"
        )  # CPU tensors — training loop moves to GPU

        labels = enc["input_ids"].clone()

        # ── Per-sample label masking ───────────────────────────────────────
        fallback_count = 0
        for i, s in enumerate(batch):
            # Tokenise the answer string alone (no special tokens)
            ans_ids = processor.tokenizer(
                s["answer"], add_special_tokens=False
            )["input_ids"]

            input_ids_list = enc["input_ids"][i].tolist()
            start = find_subseq(input_ids_list, ans_ids)

            if start is not None:
                # Mask everything before the answer tokens
                labels[i, :start] = -100
            else:
                # Answer not found verbatim (tokeniser edge case).
                # Fall back: mask everything — skip this sample's loss.
                labels[i, :] = -100
                fallback_count += 1

            # Always mask padding positions
            labels[i][enc["attention_mask"][i] == 0] = -100

        if fallback_count:
            print(f"  [collate] WARNING: {fallback_count}/{len(batch)} samples "
                  "had no answer match — masked entirely.")

        enc["labels"] = labels
        return enc

    return collate

print("Collate: OK")

Collate: OK


In [8]:
# ── Cell 7: Load model ─────────────────────────────────────────────────────
print("Loading processor …")
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)

print("Loading model (4-bit QLoRA) …")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)

# ── VRAM OPTIMIZATIONS ──
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable()  # <-- Crucial for Rank 32 on T4 GPU

print("Attaching LoRA …")
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,         # Make sure this is 32 in Cell 3!
    lora_alpha=LORA_ALPHA, # Make sure this is 64 in Cell 3!
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
))

model.print_trainable_parameters()

Loading processor …


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Attaching LoRA …
trainable params: 95,178,752 || all params: 8,387,345,408 || trainable%: 1.1348


In [9]:
# ── Cell 8: DataLoaders ────────────────────────────────────────────────────
print("\nBuilding datasets …")
collate_fn    = make_collate(processor)
train_dataset = VQAMedDataset(TRAIN_TXT, TRAIN_IMGS, "train")
val_dataset   = VQAMedDataset(VAL_TXT,   VAL_IMGS,   "val")

train_loader = DataLoader(
    train_dataset, batch_size=1, shuffle=True,
    collate_fn=collate_fn, num_workers=0, pin_memory=False)

# Fast val loader: 100-sample subset used during training for loss tracking
val_loader_fast = DataLoader(
    torch.utils.data.Subset(val_dataset, range(100)),
    batch_size=1, shuffle=False,
    collate_fn=collate_fn, num_workers=0, pin_memory=False)


Building datasets …
  [train] Loaded 4500 samples
  [val] Loaded 500 samples


In [10]:
# ── Cell 9: Sanity-check — verify label masking is working ─────────────────
#
# Each sample should have a SMALL number of active label tokens (2–8).
# If you see active == total_tokens the mask was not applied.
# If you see active == 0 the answer sub-sequence was not found.

print("\nSanity-checking collate + label masking …")
t0    = time.time()
batch = next(iter(train_loader))
print(f"  Batch shape : {batch['input_ids'].shape}")
print(f"  Batch time  : {time.time()-t0:.1f}s")

for i in range(min(3, batch["labels"].shape[0])):
    active = (batch["labels"][i] != -100).sum().item()
    total  = batch["input_ids"][i].shape[0]
    # Decode the active tokens so we can verify they are the answer
    active_ids = batch["input_ids"][i][batch["labels"][i] != -100].tolist()
    decoded    = processor.tokenizer.decode(active_ids, skip_special_tokens=False)
    print(f"  Sample {i}: {active}/{total} active label tokens  →  '{decoded}'")

assert (batch["labels"] != -100).any(), "All labels are masked — check collate!"
print("  ✓ Label masking OK\n")


Sanity-checking collate + label masking …
  Batch shape : torch.Size([1, 184])
  Batch time  : 0.2s
  Sample 0: 4/184 active label tokens  →  'spine<|im_end|>
'
  ✓ Label masking OK



In [11]:
# ── Cell 10: Optimizer & scheduler ────────────────────────────────────────
from bitsandbytes.optim import PagedAdamW8bit

# Using the 8-bit optimizer to save massive amounts of VRAM
optimizer = PagedAdamW8bit(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE
)

total_steps = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print(f"Total optimiser steps : {total_steps}")
print(f"Steps per epoch       : {len(train_loader) // GRAD_ACCUM}")

Total optimiser steps : 1405
Steps per epoch       : 281


In [12]:
# ── Cell 11: Training loop ─────────────────────────────────────────────────
#
# Checkpoint selection: save the epoch with the LOWEST val loss.
# Val loss now measures the correct thing (answer token cross-entropy)
# thanks to the fixed collate label masking.

print("\nStarting training …\n")
best_val_loss = float("inf")
log_every     = 25  # print rolling loss every N optimizer steps

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Train ───────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    opt_step     = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]")
    for step, batch in enumerate(pbar, 1):
        batch   = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss    = outputs.loss / GRAD_ACCUM
        loss.backward()
        running_loss += outputs.loss.item()

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            opt_step += 1

            if opt_step % log_every == 0:
                avg = running_loss / (GRAD_ACCUM * log_every)
                pbar.set_postfix(
                    loss=f"{avg:.4f}",
                    lr=f"{scheduler.get_last_lr()[0]:.2e}")
                running_loss = 0.0

    # ── Validation (loss only — fast 100-sample subset) ─────────────────
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader_fast,
                          desc=f"Epoch {epoch}/{NUM_EPOCHS} [val]  "):
            batch      = {k: v.to(DEVICE) for k, v in batch.items()}
            val_loss  += model(**batch).loss.item()
            val_steps += 1

    avg_val = val_loss / max(val_steps, 1)
    print(f"\n  Epoch {epoch} — val loss: {avg_val:.4f}")

    # Always save every epoch; keep track of best
    ckpt = os.path.join(OUTPUT_DIR, f"checkpoint_epoch{epoch}")
    model.save_pretrained(ckpt)
    tag  = ""
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        # Write a pointer file so we always know which checkpoint is best
        with open(os.path.join(OUTPUT_DIR, "best_checkpoint.txt"), "w") as f:
            f.write(ckpt)
        tag = "  ← best"
    print(f"  ✓ Saved → {ckpt}{tag}")

print("\nTraining complete.")


Starting training …



Epoch 1/5 [val]  : 100%|██████████| 100/100 [00:54<00:00,  1.84it/s]



  Epoch 1 — val loss: 0.6784
  ✓ Saved → /kaggle/working/qwen_vqa_lora/checkpoint_epoch1  ← best


Epoch 2/5 [val]  : 100%|██████████| 100/100 [00:53<00:00,  1.87it/s]



  Epoch 2 — val loss: 0.5645
  ✓ Saved → /kaggle/working/qwen_vqa_lora/checkpoint_epoch2  ← best


Epoch 3/5 [val]  : 100%|██████████| 100/100 [00:52<00:00,  1.91it/s]



  Epoch 3 — val loss: 0.4696
  ✓ Saved → /kaggle/working/qwen_vqa_lora/checkpoint_epoch3  ← best


Epoch 4/5 [val]  : 100%|██████████| 100/100 [00:51<00:00,  1.95it/s]



  Epoch 4 — val loss: 0.4528
  ✓ Saved → /kaggle/working/qwen_vqa_lora/checkpoint_epoch4  ← best


Epoch 5/5 [val]  : 100%|██████████| 100/100 [00:51<00:00,  1.95it/s]



  Epoch 5 — val loss: 0.5170
  ✓ Saved → /kaggle/working/qwen_vqa_lora/checkpoint_epoch5

Training complete.


In [13]:
# ── Cell 12: Free training VRAM before evaluation ─────────────────────────
import gc

print("Freeing training model from VRAM …")
del model, optimizer, scheduler
gc.collect()
torch.cuda.empty_cache()
print(f"  VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

Freeing training model from VRAM …
  VRAM free: 13.3 GB


In [14]:
# ── Cell 13: Pick best checkpoint ─────────────────────────────────────────
base_dir = OUTPUT_DIR
print("Available checkpoints:")
for f in sorted(os.listdir(base_dir)):
    print(f"  {f}")

# Prefer the pointer written by the training loop
pointer = os.path.join(base_dir, "best_checkpoint.txt")
if os.path.exists(pointer):
    with open(pointer) as fh:
        adapter_path = fh.read().strip()
    print(f"\n✓ Using best checkpoint: {adapter_path}")
else:
    # Fallback: try epoch3 → epoch2 → epoch1
    for candidate in ["checkpoint_epoch3", "checkpoint_epoch2", "checkpoint_epoch1"]:
        p = os.path.join(base_dir, candidate)
        if os.path.exists(p):
            adapter_path = p
            print(f"\n✓ Using: {adapter_path}")
            break
    else:
        raise FileNotFoundError(f"No checkpoint found in {base_dir}")

Available checkpoints:
  best_checkpoint.txt
  checkpoint_epoch1
  checkpoint_epoch2
  checkpoint_epoch3
  checkpoint_epoch4
  checkpoint_epoch5

✓ Using best checkpoint: /kaggle/working/qwen_vqa_lora/checkpoint_epoch4


In [15]:
# ── Cell 14: Load fine-tuned model for evaluation ─────────────────────────
print("Loading processor …")
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)

print("Loading base model (4-bit) …")
bnb_eval = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_eval,
    device_map="auto",
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)

print("Loading LoRA adapter …")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("✓ Model ready\n")

Loading processor …
Loading base model (4-bit) …


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loading LoRA adapter …
✓ Model ready



In [16]:
# ── Cell 15: Evaluation loop ───────────────────────────────────────────────
from transformers import AutoProcessor
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pandas as pd
import os
import torch
from PIL import Image
from tqdm import tqdm

# 1. Ensure processor is loaded into memory to avoid NameError
processor = AutoProcessor.from_pretrained(
    MODEL_ID, 
    min_pixels=MIN_PIXELS, 
    max_pixels=MAX_PIXELS
)

val_df   = pd.read_csv(VAL_TXT, sep="|",
                       names=["image_id", "question", "answer"], header=None)
smoothie = SmoothingFunction().method1

results      = []
total        = 0
exact_sum    = 0
bleu1_sum    = 0.0
bleu2_sum    = 0.0
bleu3_sum    = 0.0
bleu4_sum    = 0.0
bleu_cum_sum = 0.0
token_f1_sum = 0.0
  
def compute_bleu_ngrams(pred_tokens, gt_tokens, smoothfn):
    refs = [gt_tokens]
    hyp  = pred_tokens
    
    # ── INDIVIDUAL N-GRAM PRECISIONS ──
    b1 = sentence_bleu(refs, hyp, weights=(1.0, 0.0, 0.0, 0.0), smoothing_function=smoothfn)
    b2 = sentence_bleu(refs, hyp, weights=(0.0, 1.0, 0.0, 0.0), smoothing_function=smoothfn)
    b3 = sentence_bleu(refs, hyp, weights=(0.0, 0.0, 1.0, 0.0), smoothing_function=smoothfn)
    b4 = sentence_bleu(refs, hyp, weights=(0.0, 0.0, 0.0, 1.0), smoothing_function=smoothfn)
    
    # NLTK's Cumulative BLEU (kept for logging, though we recalculate the corpus-level one later)
    bcum = sentence_bleu(refs, hyp, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothfn)
    
    return b1, b2, b3, b4, bcum

def token_f1(pred, gt):
    pred_toks = normalise(pred).split()
    gt_toks   = normalise(gt).split()
    common    = set(pred_toks) & set(gt_toks)
    if not common:
        return 0.0
    p = len(common) / len(pred_toks)
    r = len(common) / len(gt_toks)
    return 2 * p * r / (p + r)

print(f"Evaluating on {len(val_df)} samples …\n")

for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    img_id     = str(row["image_id"]).strip()
    question   = str(row["question"]).strip()
    raw_answer = str(row["answer"]).strip()
    primary_gt = get_synonyms(raw_answer)[0]

    fname    = img_id if img_id.endswith((".jpg", ".png")) else img_id + ".jpg"
    img_path = os.path.join(VAL_IMGS, fname)
    if not os.path.exists(img_path):
        continue

    image  = Image.open(img_path).convert("RGB")
    msgs   = [
        {"role": "system",
         "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",
         "content": [{"type": "image", "image": image},
                     {"type": "text",  "text": "Question: " + question}]},
    ]
    text   = processor.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text], images=[image],
        padding=True, return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        # 2. FIXED: do_sample=False enforces greedy decoding, bypassing the CUDA float16 crash
        gen     = ft_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
        pred    = normalise(
            processor.batch_decode(trimmed, skip_special_tokens=True)[0])

    em               = exact_match_any(pred, raw_answer)
    b1, b2, b3, b4, bcum = compute_bleu_ngrams(
        pred.split(), primary_gt.split(), smoothie)
    tf1              = token_f1(pred, primary_gt)

    total        += 1
    exact_sum    += em
    bleu1_sum    += b1
    bleu2_sum    += b2
    bleu3_sum    += b3
    bleu4_sum    += b4
    bleu_cum_sum += bcum
    token_f1_sum += tf1

    results.append({
        "Image ID":        img_id,
        "Question":        question,
        "Ground Truth":    raw_answer,
        "Prediction":      pred,
        "Exact Match":     em,
        "BLEU-1":          round(b1,   4),
        "BLEU-2":          round(b2,   4),
        "BLEU-3":          round(b3,   4),
        "BLEU-4":          round(b4,   4),
        "Cumulative BLEU": round(bcum, 4),
        "Token F1":        round(tf1,  4),
    })

# ── Save ──────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
results_df.to_csv(EVAL_CSV, index=False)
print(f"\nResults → {EVAL_CSV}")

# ── Aggregate metrics (averaged over all samples) ─────────────────────────
avg_b1  = bleu1_sum    / total * 100
avg_b2  = bleu2_sum    / total * 100
avg_b3  = bleu3_sum    / total * 100
avg_b4  = bleu4_sum    / total * 100
avg_f1  = token_f1_sum / total * 100
avg_em  = exact_sum    / total * 100

# ── Cumulative BLEU — geometric mean of the FOUR AGGREGATE BLEU-n scores ──
# This matches Lameesa et al.'s definition and reported numbers.
cumulative_bleu = (avg_b1 * avg_b2 * avg_b3 * avg_b4) ** 0.25

# ── Report (scale ×100 to match Lameesa et al. Table 2 format) ───────────
print("\n" + "=" * 57)
print("   FINE-TUNED — VQA-MED 2021 RESULTS")
print("=" * 57)
print(f"  Checkpoint     : {adapter_path}")
print(f"  Total          : {total}")
print(f"  Exact accuracy : {avg_em:.2f}%")
print(f"  BLEU-1         : {avg_b1:.2f}")
print(f"  BLEU-2         : {avg_b2:.2f}")
print(f"  BLEU-3         : {avg_b3:.2f}")
print(f"  BLEU-4         : {avg_b4:.2f}")
print(f"  Cumulative BLEU: {cumulative_bleu:.2f}    <- Geometric Mean")
print(f"  Avg Token F1   : {avg_f1:.2f}")
print("=" * 57)


Evaluating on 500 samples …



100%|██████████| 500/500 [10:32<00:00,  1.27s/it]


Results → /kaggle/working/finetuned_model_metrics.csv

   FINE-TUNED — VQA-MED 2021 RESULTS
  Checkpoint     : /kaggle/working/qwen_vqa_lora/checkpoint_epoch4
  Total          : 500
  Exact accuracy : 41.80%
  BLEU-1         : 43.65
  BLEU-2         : 33.27
  BLEU-3         : 17.75
  BLEU-4         : 10.18
  Cumulative BLEU: 22.63    <- Geometric Mean
  Avg Token F1   : 44.07


In [17]:
# ── Cell 16: Sample predictions ───────────────────────────────────────────
print("\n── 20 sample predictions ──")
sample = results_df.sample(min(20, len(results_df)), random_state=42)
for _, r in sample.iterrows():
    mark = "✓" if r["Exact Match"] else "✗"
    print(f"  {mark}  GT: {str(r['Ground Truth']):<40}  "
          f"Pred: {r['Prediction']:<35}  "
          f"F1={r['Token F1']:.2f}")


── 20 sample predictions ──
  ✗  GT: lymphangitic carcinomatosis               Pred: cystic fibrosis                      F1=0.00
  ✓  GT: rheumatoid arthritis                      Pred: rheumatoid arthritis                 F1=1.00
  ✗  GT: odontoid fracture                         Pred: cervical spine fracture (unstable) with cord injury  F1=0.22
  ✗  GT: intraosseous lipoma                       Pred: osteomyelitis                        F1=0.00
  ✗  GT: pulmonary embolism                        Pred: aortic dissection                    F1=0.00
  ✓  GT: motor vehicle accidents and instrument failure  Pred: motor vehicle accidents and instrument failure  F1=1.00
  ✗  GT: round atelectasis                         Pred: pulmonary embolism                   F1=0.00
  ✓  GT: classic bosniak class iii complex renal cyst  Pred: classic bosniak class iii complex renal cyst  F1=1.00
  ✗  GT: congenital cystic adenomatoid malformation  Pred: osteosarcoma                         F1=0.00
  ✗  